# Entrenamiento final – El Crustáceo Cascarudo (MBIA2602)
**Un solo paso:** `Entorno de ejecución → Cambiar tipo → GPU T4`, luego `Entorno de ejecución → Ejecutar todo`.

Este cuaderno:
1. Amplía el dataset de imágenes (Wikimedia + filtro CLIP).
2. Entrena el clasificador de intenciones (spaCy + TF-IDF + SVM).
3. Entrena el clasificador visual (ResNet50) con **división 70/15/15 y conjunto de PRUEBA independiente** (corrección del profesor).
4. Corre pruebas automáticas, evaluación del chatbot y benchmark.
5. Guarda modelos, métricas y evidencias en `Mi unidad/crustaceo_entrega/`.

In [ ]:
# 1) Montar Google Drive y localizar el dataset
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)
DATA = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'dataset_crustaceo' in dirs:
        DATA = os.path.join(root, 'dataset_crustaceo'); break
assert DATA, "No encontré la carpeta dataset_crustaceo en tu Drive"
print("Dataset:", DATA)
for c in sorted(os.listdir(DATA)):
    if os.path.isdir(os.path.join(DATA, c)):
        print(f"  {c}: {len(os.listdir(os.path.join(DATA, c)))} imágenes")

In [ ]:
# 2) Descargar el código del proyecto e instalar dependencias (versiones iguales a la nube)
%cd /content
!rm -rf chatbot-pedidos-crustaceo
!git clone -q https://github.com/CtrlAltDoc/chatbot-pedidos-crustaceo.git
%cd /content/chatbot-pedidos-crustaceo
!pip -q install scikit-learn==1.6.1 "spacy>=3.7,<3.9" https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl

In [ ]:
# 2b) Ampliar el dataset con fotos libres de Wikimedia Commons (corrección P2)
# Agrega fotos hasta llegar a 75 por clase, sin borrar las tuyas. CLIP descarta lo que no sea el producto.
# Reanudable: si se corta, vuelve a correr esta celda. Créditos en dataset_crustaceo/creditos_wikimedia.csv
!python -m scripts.descargar_dataset --data "$DATA" --meta 75

In [ ]:
# 3) Clasificador de intenciones (NLP)
!python -m scripts.train_nlp

In [ ]:
# 4) Clasificador visual (ResNet50 + transferencia de aprendizaje, test independiente)
!python -m scripts.train_vision --data "$DATA" 

In [ ]:
# 5) Pruebas automáticas (pytest)
!python -m pytest -q 2>&1 | tee docs/pruebas_pytest.txt

In [ ]:
# 6) Pruebas funcionales y métricas del chatbot (usa imágenes del conjunto de prueba)
!python -m scripts.eval_chatbot --images "$DATA" 

In [ ]:
# 7) Benchmark de desempeño / escalabilidad
import json, glob
test_imgs = json.load(open('models/split_vision.json'))['test']
!python -m scripts.benchmark --image "{test_imgs[0]}" 

In [ ]:
# 8) Guardar todo en Drive: Mi unidad/crustaceo_entrega/
OUT = '/content/drive/MyDrive/crustaceo_entrega'
!mkdir -p "$OUT"
!cp -r models docs "$OUT"/
print("Listo. Archivos en", OUT)
!ls -R "$OUT" | head -40

In [ ]:
# 9) Ver resultados
from IPython.display import Image, display
for f in ['docs/metricas/confusion_vision.png', 'docs/metricas/confusion_nlp.png']:
    display(Image(f, width=480))
print(json.dumps(json.load(open('models/metrics_vision.json'))['test'] | {'confusion_matrix': '(ver imagen)'}, indent=2, ensure_ascii=False))